# Fine-tune Whisper on CDLI Non-Standard Kenyan Speech (Noise-Robust)

This notebook fine-tunes Whisper on CDLI non-standard Kenyan speech datasets with **waveform-level audio augmentation** to improve robustness in real-world noisy environments (ambient crowd noise, GSM codec compression, room reverb).

**Changes from baseline notebook:**
- `prepare_features_augmented()` applies waveform augmentation on the training split only
- `augment_audio()` simulates: volume perturbation, Gaussian noise, GSM codec downsampling, and room reverb
- `AUGMENT_PROB` controls augmentation probability per example (default 0.5)
- Evaluation uses clean audio throughout for comparable WER/CER
- `processing_class` replaces deprecated `tokenizer` in Trainer
- Early stopping and weight decay consolidated into `Seq2SeqTrainingArguments`

## Authentication

In [1]:
from huggingface_hub import login
HF_TOKEN = input("Enter HF token: ")
login(token=HF_TOKEN)

## Settings

**Adapt these for your run. Everything else should be left as-is.**

### Directories

In [2]:
import os

LOCAL_STORAGE_DIR = '/jupyter_kernel'
BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
os.makedirs(BASE_DIR, exist_ok=True)

# Set run name -- increment for each new run
OUTPUT_DIR = os.path.join(BASE_DIR, 'whisper-small-kenyan-english-nonstandard-robust_v1_run4')
# OUTPUT_DIR = os.path.join(BASE_DIR, 'whisper-small-kenyan-swahili-nonstandard-robust_v1_run1')

print(f"Will write model to: {OUTPUT_DIR}")
if os.path.exists(OUTPUT_DIR):
    raise ValueError("Output directory already exists. Increment run number or delete existing directory.")


Will write model to: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard-robust_v1_run4


ValueError: Output directory already exists. Increment run number or delete existing directory.

### Model and Dataset Settings

In [3]:
WHISPER_MODEL_TYPE = "openai/whisper-small"
# WHISPER_MODEL_TYPE = "openai/whisper-medium"

# English non-standard
LANGUAGE = 'en'
DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v1.0"

# Swahili non-standard (uncomment to switch)
# LANGUAGE = 'sw'
# DATASET_NAME = "cdli/kenyan_swahili_nonstandard_speech_v0.9"

### Augmentation Settings

Controls waveform-level noise augmentation applied during training only.
Start with `AUGMENT_PROB = 0.5`. Increase to 0.7 in later runs if WER improves.

In [4]:
# Master switch: set to False to disable all waveform augmentation (reproduces baseline)
USE_WAVEFORM_AUGMENTATION = True

# Probability that any single augmentation is applied to a given example
# Run 3 finding: 0.4 strikes a better clean/noisy balance than 0.5 or 0.7
AUGMENT_PROB = 0.4

# Individual augmentation enable flags
AUG_VOLUME_PERTURB = True   # always mild, safe to keep on
AUG_GAUSSIAN_NOISE = True   # simulates crowd/ambient noise
AUG_GSM_CODEC      = True   # simulates mobile network compression
AUG_REVERB         = False  # simulates small room/kiosk acoustics

# Noise intensity bounds (increase cautiously for larger datasets)
NOISE_LEVEL_MIN = 0.002
NOISE_LEVEL_MAX = 0.01


### Model Architecture Settings

In [5]:
# Which parts of the model to update
UPDATE_ENCODER = True
UPDATE_PROJ    = True
UPDATE_DECODER = True  # set False to use partial decoder unfreezing below

# Partial decoder unfreezing (only applies when UPDATE_DECODER = False)
# whisper-small has 12 decoder layers (0-11). Unfreeze the last N.
NUM_DECODER_LAYERS_TO_UNFREEZE = 2

# SpecAugment (operates on log-mel spectrograms, complementary to waveform augmentation)
USE_SPECAUGMENT = True

### Trainer Settings

In [6]:
LOGGING_STEPS = 5
SAVE_STEPS    = 50   # set to 0 to only save last and best

MAX_EPOCHS = 10
MAX_STEPS  = 2000    # increase for larger datasets

LEARNING_RATE     = 3e-6
LR_SCHEDULER_TYPE = 'polynomial'   # 'constant_with_warmup' or 'polynomial'
LR_WARMUP_STEPS   = 100
LR_END            = 1e-8
LR_DECAY_POWER    = 1

WEIGHT_DECAY            = 0.01
EARLY_STOPPING_PATIENCE = 7

BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 16

MAX_GEN_LEN   = 128
EVAL_ON_START = True
EVAL_STEPS    = 50

USE_FP16 = True
USE_BF16 = False   # enable for A100/A40

NUM_CHECKPOINTS_TO_STORE = 2

# Run 4: use noise-augmented dev set as the early stopping signal.
# Set to False to revert to clean dev WER (Run 3 behaviour).
USE_NOISY_DEV_FOR_EARLY_STOPPING = True

# Don't change these
TASK            = "transcribe"
BASE_MODEL_NAME = WHISPER_MODEL_TYPE
print(f"Base model: {BASE_MODEL_NAME} | Language: {LANGUAGE} | Dataset: {DATASET_NAME}")
print(f"Waveform augmentation: {USE_WAVEFORM_AUGMENTATION} | Augment prob: {AUGMENT_PROB}")
print(f"Noisy dev early stopping: {USE_NOISY_DEV_FOR_EARLY_STOPPING}")


Base model: openai/whisper-small | Language: en | Dataset: cdli/kenyan_english_nonstandard_speech_v1.0
Waveform augmentation: True | Augment prob: 0.4
Noisy dev early stopping: True


## Imports and Environment Setup

In [7]:
import random
import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import datasets
import evaluate
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from huggingface_hub import hf_hub_download
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Disable dataset caching (saves disk on Modal volumes)
datasets.disable_caching()
print('Dataset caching:', datasets.is_caching_enabled())

# Avoid thread contention with multiprocessing map
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

num_proc = min(32, os.cpu_count())
print(f"CPU workers for dataset mapping: {num_proc}")

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
transcript_normalizer = BasicTextNormalizer()

Dataset caching: False
Device: cuda
CPU workers for dataset mapping: 20


## Waveform Augmentation

Applied to training audio only, before log-mel feature extraction.
Simulates four real-world Kenyan deployment conditions:

1. **Volume perturbation** — microphone gain variation across devices
2. **Gaussian noise** — ambient crowd noise, wind, market environments
3. **GSM codec simulation** — 8kHz downsample/resample to mimic mobile network compression
4. **Room reverb** — small kiosk/office acoustic echo

Evaluation splits always use clean audio for reproducible WER/CER comparisons.

In [8]:
def augment_audio(audio_array: np.ndarray, sample_rate: int, apply_prob: float = 0.5) -> np.ndarray:
    """
    Apply waveform-level augmentations to simulate Kenyan deployment conditions.

    Args:
        audio_array: Raw audio waveform as numpy array.
        sample_rate: Audio sample rate (typically 16000 Hz).
        apply_prob: Probability that each stochastic augmentation is applied.

    Returns:
        Augmented waveform as numpy array, same shape as input.
    """
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)  # (1, T)

    # 1. Volume perturbation (always applied, mild range)
    if AUG_VOLUME_PERTURB:
        gain = random.uniform(0.7, 1.3)
        waveform = waveform * gain

    # 2. Gaussian noise (crowd/ambient)
    if AUG_GAUSSIAN_NOISE and random.random() < apply_prob:
        noise_level = random.uniform(NOISE_LEVEL_MIN, NOISE_LEVEL_MAX)
        noise = torch.randn_like(waveform) * noise_level
        waveform = waveform + noise

    # 3. GSM codec simulation (mobile network compression artifact)
    # Downsample to 8kHz then resample back to original rate
    if AUG_GSM_CODEC and random.random() < apply_prob:
        resample_down = T.Resample(orig_freq=sample_rate, new_freq=8000)
        resample_up   = T.Resample(orig_freq=8000, new_freq=sample_rate)
        waveform = resample_up(resample_down(waveform))

    # 4. Room reverb (small kiosk/office acoustic echo)
    # Implemented as a short delay blend; lower prob since it's more disruptive
    if AUG_REVERB and random.random() < apply_prob * 0.5:
        reverb_gain   = random.uniform(0.1, 0.25)
        delay_samples = random.randint(int(0.01 * sample_rate), int(0.05 * sample_rate))
        delayed = torch.zeros_like(waveform)
        delayed[:, delay_samples:] = waveform[:, :-delay_samples]
        waveform = waveform + reverb_gain * delayed

    # Clip to prevent saturation from compounded augmentations
    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform.squeeze(0).numpy()

## Helper Functions

In [9]:
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_dataset_split(dataset_name: str, split: str, limit_to_30_seconds: bool = True):
    """Load a HF dataset split and optionally filter to Whisper's 30s context window."""
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda ex: ex['audio_length'] <= 30)
        print(f"[{split}] Filtered {orig_len} -> {len(ds)} examples (<= 30s)")
    return ds


def is_valid_label_length(example):
    """Exclude examples with token sequences exceeding Whisper's decoder limit."""
    if 'labels' not in example:
        return False
    valid_labels = [l for l in example['labels'] if l != -100]
    return len(valid_labels) <= 448


def get_wer(references, predictions, normalize=True, verbose=True):
    rs, ps = references, predictions
    if normalize:
        ps = [transcript_normalizer(x) for x in predictions]
        rs = [transcript_normalizer(x) for x in references]
    if verbose:
        for r, p in zip(rs, ps):
            print(f"REF: {r}")
            print(f"HYP: {p}")
            print()
    return wer_metric.compute(references=rs, predictions=ps)


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_strs  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wers, cers = [], []
    for pred_str, label_str in zip(pred_strs, label_strs):
        p = transcript_normalizer(pred_str)
        l = transcript_normalizer(label_str)
        wers.append(wer_metric.compute(predictions=[p], references=[l]))
        cers.append(cer_metric.compute(predictions=[p], references=[l]))

    wer = np.mean([min(1.0, x) for x in wers])
    cer = np.mean([min(1.0, x) for x in cers])
    print(f'WER (adjusted): {wer:.4f} | CER (adjusted): {cer:.4f}')
    return {"wer": wer, "cer": cer}

## Feature Extraction Functions

Two variants:
- `prepare_features` — clean extraction for eval/test splits
- `prepare_features_augmented` — waveform augmentation then extraction for training split

In [10]:
print(f"Loading processor for: {WHISPER_MODEL_TYPE} | language: {LANGUAGE}")
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)


def prepare_features(example):
    """Clean feature extraction. Use for dev and test splits."""
    example["input_features"] = processor.feature_extractor(
        example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"]
    ).input_features[0]
    example["labels"]       = processor.tokenizer(example["transcription"]).input_ids
    example["token_length"] = len(example["labels"])
    return example


def prepare_features_augmented(example):
    """
    Waveform augmentation then feature extraction. Use for training split only.
    Falls back to clean extraction if USE_WAVEFORM_AUGMENTATION is False.
    """
    audio_array = example["audio"]["array"]
    sample_rate = example["audio"]["sampling_rate"]

    if USE_WAVEFORM_AUGMENTATION:
        audio_array = augment_audio(audio_array, sample_rate, apply_prob=AUGMENT_PROB)

    example["input_features"] = processor.feature_extractor(
        audio_array, sampling_rate=sample_rate
    ).input_features[0]
    example["labels"]       = processor.tokenizer(example["transcription"]).input_ids
    example["token_length"] = len(example["labels"])
    return example

Loading processor for: openai/whisper-small | language: en


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

## Data Collator

In [11]:
# Note: "The attention mask is not set..." warning on this collator can be safely ignored.
# See: https://discuss.huggingface.co/t/finetuning-whisper-attention-mask-not-set-and-canot-be-inferred/97456

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    # Alias so newer Trainer versions don't fall back to the deprecated path
    @property
    def tokenizer(self):
        return self.processor.tokenizer

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels         = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

## Load and Prepare Datasets

Training split uses `prepare_features_augmented`.
Dev and test splits use `prepare_features` (clean audio) for reporting.

Run 4 addition: a noise-augmented mirror of the dev set (`noisy_dev_dataset`) is built
using the same `prepare_features_augmented` pipeline. This is passed to the Trainer as
`eval_dataset` so that early stopping and best-checkpoint selection are based on noisy
dev WER rather than clean dev WER. Clean dev and test evaluations are run separately
after training for reporting.


In [14]:
train_dataset = load_dataset_split(DATASET_NAME, split='train', limit_to_30_seconds=True)
train_dataset = train_dataset.map(
    prepare_features_augmented,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Train examples after processing: {len(train_dataset)}")


Generating test split:   0%|          | 0/928 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/542 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4378 [00:00<?, ? examples/s]

Generating extra split:   0%|          | 0/150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4378 [00:00<?, ? examples/s]

[train] Filtered 4378 -> 4243 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/4243 [00:00<?, ? examples/s]

Train examples after processing: 4243


In [15]:
# Clean dev set — used for post-training reporting only
dev_dataset = load_dataset_split(DATASET_NAME, split='validation', limit_to_30_seconds=True)
dev_dataset = dev_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Dev examples after processing: {len(dev_dataset)}")

# Noise-augmented dev set — used as the Trainer early stopping signal (Run 4)
if USE_NOISY_DEV_FOR_EARLY_STOPPING:
    noisy_dev_dataset = load_dataset_split(DATASET_NAME, split='validation', limit_to_30_seconds=True)
    noisy_dev_dataset = noisy_dev_dataset.map(
        prepare_features_augmented,
        remove_columns=['audio'],
        writer_batch_size=1,
        num_proc=num_proc
    )
    print(f"Noisy dev examples after processing: {len(noisy_dev_dataset)}")
    eval_dataset_for_trainer = noisy_dev_dataset
else:
    eval_dataset_for_trainer = dev_dataset

print(f"Early stopping signal: {'noisy dev' if USE_NOISY_DEV_FOR_EARLY_STOPPING else 'clean dev'}")


Filter:   0%|          | 0/542 [00:00<?, ? examples/s]

[validation] Filtered 542 -> 542 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/542 [00:00<?, ? examples/s]

Dev examples after processing: 542


Filter:   0%|          | 0/542 [00:00<?, ? examples/s]

[validation] Filtered 542 -> 542 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/542 [00:00<?, ? examples/s]

Noisy dev examples after processing: 542
Early stopping signal: noisy dev


In [17]:
test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
test_dataset = test_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Test examples after processing: {len(test_dataset)}")


Filter:   0%|          | 0/928 [00:00<?, ? examples/s]

[test] Filtered 928 -> 926 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/926 [00:00<?, ? examples/s]

Test examples after processing: 926


## Load and Configure Model

In [18]:
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
base_model = base_model.to(device)

# Task and language
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task     = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.forced_decoder_ids  = None
base_model.config.use_cache           = False  # required for gradient checkpointing

print(f"Model loaded: {WHISPER_MODEL_TYPE}")
print(f"Language: {base_model.generation_config.language}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded: openai/whisper-small
Language: en


In [19]:
if USE_SPECAUGMENT:
    base_model.config.apply_spec_augment     = True
    base_model.config.mask_time_prob         = 0.05
    base_model.config.mask_time_length       = 10
    base_model.config.mask_time_min_masks    = 2
    base_model.config.mask_feature_prob      = 0.05
    base_model.config.mask_feature_length    = 10
    base_model.config.mask_feature_min_masks = 2

print(f"SpecAugment: {base_model.config.apply_spec_augment}")

SpecAugment: True


### Layer Freezing

Run **one** of the two cells below depending on your `UPDATE_DECODER` setting.

In [20]:
# Full unfreeze (run this when UPDATE_DECODER = True)
if UPDATE_DECODER:
    base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
    base_model.model.decoder.requires_grad_(True)
    base_model.proj_out.requires_grad_(UPDATE_PROJ)

    print(f"Encoder params: {count_trainable_parameters(base_model.model.encoder):,} / {base_model.model.encoder.num_parameters():,}")
    print(f"Decoder params: {count_trainable_parameters(base_model.model.decoder):,} / {base_model.model.decoder.num_parameters():,}")
    print(f"Total trainable: {count_trainable_parameters(base_model):,} / {base_model.model.num_parameters():,}")

Encoder params: 88,154,112 / 88,154,112
Decoder params: 153,580,800 / 153,580,800
Total trainable: 241,734,912 / 241,734,912


In [21]:
# Partial decoder unfreeze (run this when UPDATE_DECODER = False)
if not UPDATE_DECODER:
    base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
    base_model.proj_out.requires_grad_(UPDATE_PROJ)

    # Freeze full decoder, then selectively unfreeze last N layers
    base_model.model.decoder.requires_grad_(False)
    for layer in base_model.model.decoder.layers[-NUM_DECODER_LAYERS_TO_UNFREEZE:]:
        layer.requires_grad_(True)

    print(f"Partial unfreeze: last {NUM_DECODER_LAYERS_TO_UNFREEZE} decoder layers")
    print(f"Encoder params: {count_trainable_parameters(base_model.model.encoder):,} / {base_model.model.encoder.num_parameters():,}")
    print(f"Decoder params: {count_trainable_parameters(base_model.model.decoder):,} / {base_model.model.decoder.num_parameters():,}")
    print(f"Total trainable: {count_trainable_parameters(base_model):,} / {base_model.model.num_parameters():,}")

## Configure Trainer

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    include_num_input_tokens_seen=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    push_to_hub=False,
    remove_unused_columns=False,
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    per_device_train_batch_size=BATCH_SIZE,
    eval_on_start=EVAL_ON_START,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    generation_max_length=MAX_GEN_LEN,
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    lr_scheduler_kwargs={
        "lr_end": LR_END,
        "power": LR_DECAY_POWER,
    },
    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
)

print(f"Trainer configured. Output: {OUTPUT_DIR}")
print(f"Weight decay: {training_args.weight_decay} | Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Eval dataset: {'noisy dev' if USE_NOISY_DEV_FOR_EARLY_STOPPING else 'clean dev'}")


Trainer configured. Output: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard-robust_v1_run4
Weight decay: 0.01 | Early stopping patience: 7
Eval dataset: noisy dev


In [23]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=base_model.config.decoder_start_token_id,
)

# eval_dataset_for_trainer: noisy dev when USE_NOISY_DEV_FOR_EARLY_STOPPING=True,
# clean dev otherwise. Early stopping and best-checkpoint selection act on this dataset.
trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)


[RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Run Training

TensorBoard logs write to `OUTPUT_DIR/logs`. Use `tensorboard_server.py` on Modal to view.

Run the model dir print below first to copy the path.

In [24]:
print('Training dir (for TensorBoard):', OUTPUT_DIR)

Training dir (for TensorBoard): /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard-robust_v1_run4


In [25]:
# Fresh training run
#trainer.train()

# Resume from checkpoint if interrupted:
trainer.train(resume_from_checkpoint=True)

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Step,Training Loss,Validation Loss,Wer,Cer,Input Tokens Seen
1550,0.573700,0.901335,0.268447,0.172278,11869680000
1600,0.749100,0.896379,0.271355,0.174710,12250560000
1650,0.575300,0.898102,0.271496,0.173573,12634560000
1700,0.622300,0.896813,0.270799,0.173112,13018560000
1750,0.677000,0.896737,0.270318,0.173726,13399440000
1800,0.656700,0.898071,0.270196,0.172606,13783440000
1850,0.652600,0.897107,0.270209,0.172154,14167440000
1900,0.564300,0.896879,0.270831,0.173812,14548320000
1950,0.639300,0.895913,0.270662,0.173584,14932320000
2000,0.673400,0.896393,0.269754,0.172454,15313200000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

WER (adjusted): 0.2684 | CER (adjusted): 0.1723


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

...

WER (adjusted): 0.2707 | CER (adjusted): 0.1736


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

WER (adjusted): 0.2698 | CER (adjusted): 0.1725


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=2000, training_loss=0.14529199397563936, metrics={'train_runtime': 4905.5471, 'train_samples_per_second': 13.046, 'train_steps_per_second': 0.408, 'total_flos': 1.84131914674176e+19, 'train_loss': 0.14529199397563936, 'epoch': 15.037593984962406, 'num_input_tokens_seen': 15313200000})

## Evaluate

Both splits use clean (unaugmented) audio. WER/CER here reflects real-world clean speech performance.

Note: the Trainer's internal eval during training ran on `eval_dataset_for_trainer` (noisy dev in Run 4).
The cells below evaluate the best checkpoint against clean dev and clean test — these are the
reporting numbers for the run report.


In [26]:
# Clean dev — reporting only (Trainer used noisy dev internally)
print("--- Dev set evaluation (clean) ---")
trainer.evaluate(dev_dataset, language=LANGUAGE)


--- Dev set evaluation (clean) ---


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

WER (adjusted): 0.2174 | CER (adjusted): 0.1363


{'eval_loss': 0.7948426604270935,
 'eval_wer': 0.21739420526007974,
 'eval_cer': 0.13627951761324064,
 'eval_runtime': 195.095,
 'eval_samples_per_second': 2.778,
 'eval_steps_per_second': 0.174,
 'epoch': 15.037593984962406,
 'num_input_tokens_seen': 15313200000}

In [27]:
print("--- Test set evaluation (clean) ---")
trainer.evaluate(test_dataset, language=LANGUAGE)


--- Test set evaluation (clean) ---


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

WER (adjusted): 0.1569 | CER (adjusted): 0.0940


{'eval_loss': 0.6703212261199951,
 'eval_wer': 0.1569246641490259,
 'eval_cer': 0.09402627415936614,
 'eval_runtime': 328.5424,
 'eval_samples_per_second': 2.819,
 'eval_steps_per_second': 0.177,
 'epoch': 15.037593984962406,
 'num_input_tokens_seen': 15313200000}

## Evaluate on Noise-Augmented Test Set

Runs the same test split through the augmentation pipeline before evaluation.
Comparing these results against the clean test WER quantifies noise robustness.

In Run 4, the best checkpoint was selected based on noisy dev WER, so this evaluation
reflects a checkpoint that was explicitly optimised for noisy conditions — unlike Runs 1–3
where the checkpoint was selected on clean dev WER.

Publish clean test WER, noisy test WER, and the gap (robustness delta) in the model card.


In [28]:
# Reload raw test split and apply augmented feature extraction
noisy_test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
noisy_test_dataset = noisy_test_dataset.map(
    prepare_features_augmented,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Noisy test examples: {len(noisy_test_dataset)}")

print("--- Noise-augmented test set evaluation ---")
trainer.evaluate(noisy_test_dataset, language=LANGUAGE)


Filter:   0%|          | 0/928 [00:00<?, ? examples/s]

[test] Filtered 928 -> 926 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/926 [00:00<?, ? examples/s]

Noisy test examples: 926
--- Noise-augmented test set evaluation ---


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

WER (adjusted): 0.2175 | CER (adjusted): 0.1367


{'eval_loss': 0.786375105381012,
 'eval_wer': 0.21750514187316536,
 'eval_cer': 0.13666472942488045,
 'eval_runtime': 324.9356,
 'eval_samples_per_second': 2.85,
 'eval_steps_per_second': 0.178,
 'epoch': 15.037593984962406,
 'num_input_tokens_seen': 15313200000}

## Save Model

In [29]:
# load_best_model_at_end=True means the best checkpoint is already loaded after training.
best_model_dir = os.path.join(OUTPUT_DIR, 'best_model')
print(f"Saving best model to: {best_model_dir}")
trainer.model.save_pretrained(best_model_dir, safe_serialization=True)
processor.save_pretrained(best_model_dir)
print("Done.")

Saving best model to: /jupyter_kernel/trained_models/whisper-small-kenyan-english-nonstandard-robust_v1_run4/best_model
Done.
